# Build Your Own AI Assistant: A RAG Pipeline PoC with Python & Ollama

**Liga AC LABS** — Universitatea Politehnica Timisoara, April 2026

This notebook mirrors the staged `.py` scripts in the repo. Work through each section in order.

---

## Stage 1 — Foundations & Setup

### Smoke Test
Verify that Ollama is running and both models respond.

In [ ]:
# Configuration — used throughout the notebook
OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
LANGUAGE_MODEL = "llama3.2:3b"

# Chunking parameters (mirrored from the production Java pipeline)
CHUNK_SIZE = 920
CHUNK_OVERLAP = 230

# Retrieval parameters
RETRIEVER_TOP_K = 10
RETRIEVER_SCORE_THRESHOLD = 0.3

# Paths
PDF_PATH = "corpus/Taycan_2021_EN_excerpt.pdf"  # Use excerpt for speed
CHROMA_PERSIST_DIR = "chroma_data"  # Notebook uses repo-root chroma dir
COLLECTION_NAME = "taycan_manual"

In [ ]:
# Test the language model
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model=LANGUAGE_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.0)
response = llm.invoke("Say hello in one sentence.")
print(f"LLM response: {response}")

In [ ]:
# Test the embedding model
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
vector = embeddings.embed_query("This is a test sentence.")
print(f"Embedding dimensions: {len(vector)}")
print(f"First 5 values: {vector[:5]}")

If both cells above ran without errors, your environment is ready.

---

## Stage 2 — Building the Pipeline

### Step 1: Load and Chunk the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# TODO: Load the PDF
# loader = PyPDFLoader(PDF_PATH)
# pages = loader.load()
# print(f"Loaded {len(pages)} pages.")

# TODO: Split into chunks
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=CHUNK_SIZE,
#     chunk_overlap=CHUNK_OVERLAP,
# )
# chunks = splitter.split_documents(pages)
# print(f"Created {len(chunks)} chunks.")

# TODO: Preview the first chunk
# print(chunks[0].page_content[:300])

### Step 2: Embed and Store in ChromaDB

In [ ]:
import time
from langchain_chroma import Chroma

# TODO: Create embeddings and store in ChromaDB
# embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
#
# start = time.time()
# vectorstore = Chroma.from_documents(
#     documents=chunks,
#     embedding=embeddings,
#     persist_directory=CHROMA_PERSIST_DIR,
#     collection_name=COLLECTION_NAME,
# )
# elapsed = time.time() - start
#
# count = vectorstore._collection.count()
# print(f"Stored {count} chunks in {elapsed:.1f}s")

### Step 3: Build the Retrieval QA Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

PROMPT_TEMPLATE = """Based on the following context, answer the question. \
If the answer is not in the context, say 'I don't have enough information to answer that.'

Context:
{context}

Question: {question}

Answer:"""

# TODO: Load the existing store (if you ran the cell above, vectorstore is already loaded)
# If starting fresh:
# vectorstore = Chroma(
#     persist_directory=CHROMA_PERSIST_DIR,
#     embedding_function=OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL),
#     collection_name=COLLECTION_NAME,
# )

# TODO: Build the retrieval chain
# retriever = vectorstore.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={"k": RETRIEVER_TOP_K, "score_threshold": RETRIEVER_SCORE_THRESHOLD},
# )
#
# prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
#
# chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | prompt
#     | llm
#     | StrOutputParser()
# )

In [ ]:
# TODO: Ask a question and measure performance
# import time
#
# question = "What is the maximum charging power of the Taycan?"
#
# # Measure retrieval separately
# t0 = time.perf_counter()
# retrieved_docs = retriever.invoke(question)
# t_retrieval = time.perf_counter() - t0
#
# # Measure end-to-end
# t1 = time.perf_counter()
# answer = chain.invoke(question)
# t_total = time.perf_counter() - t1
# t_generation = max(t_total - t_retrieval, 0.0)
#
# print(f"Bot: {answer}\n")
# print(f"--- Metrics ---")
# print(f"  Chunks retrieved : {len(retrieved_docs)}")
# print(f"  Retrieval time   : {t_retrieval:.3f}s")
# print(f"  Generation time  : {t_generation:.3f}s")
# print(f"  Total time       : {t_total:.3f}s")

---

## Stage 3 — Exploration & Comparison

### Experiment with different prompts

In [ ]:
# Try a concise prompt
CONCISE_PROMPT = """Answer the question in 1-2 sentences using ONLY the context below. \
If the context doesn't contain the answer, say 'Not found in the document.'

Context:
{context}

Question: {question}

Concise answer:"""

# TODO: Build a new chain with the concise prompt and compare
# concise_chain = (
#     {"context": retriever, "question": RunnablePassthrough()}
#     | ChatPromptTemplate.from_template(CONCISE_PROMPT)
#     | llm
#     | StrOutputParser()
# )
#
# question = "How do I activate the lane keeping assistant?"
# print("Default:", chain.invoke(question))
# print()
# print("Concise:", concise_chain.invoke(question))

In [ ]:
# Experiment: Change retrieval parameters
# Try fewer retrieved chunks
# small_retriever = vectorstore.as_retriever(
#     search_type="similarity_score_threshold",
#     search_kwargs={"k": 3, "score_threshold": 0.5},
# )
#
# small_chain = (
#     {"context": small_retriever, "question": RunnablePassthrough()}
#     | ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
#     | llm
#     | StrOutputParser()
# )
#
# question = "What is the recommended tire pressure?"
# print("top_k=10:", chain.invoke(question))
# print()
# print("top_k=3:", small_chain.invoke(question))

### Questions for discussion

1. What kinds of questions does the 3B model handle well? Where does it struggle?
2. How does the number of retrieved chunks affect answer quality?
3. In an automotive context (no internet), why is local inference the only option?
4. How does RAG reduce the need for larger models?

See `stage_3_exploration/06_local_vs_cloud_notes.md` for a full comparison.